# ICA Applied to Tabular Data

This notebook demonstrates Independent Component Analysis (ICA) for blind source separation in tabular data. Following the audio example, we apply a `Centering -> Whitening -> ICA` pipeline to recover statistically independent components from the observed columns.

The files in `data/` contain the same three mixtures with 100, 1,000, and 100,000 observations.

## 1. Import libs

In [1]:
from pathlib import Path

import pandas as pd
import torch

from ica.data.tabular import TabularDataset
from ica.model import Kurtosis, Negentropy
from ica.preprocessing import Centering, Whitening
from ica.utils import Pipeline

_ = torch.manual_seed(0)

## 2. Load and inspect CSV data

In [2]:
DATASET_DIR = Path("data/")
N_SAMPLES = 100_000  # also available: 100 and 1_000

dataset = TabularDataset(
    source=DATASET_DIR / f"mix_{N_SAMPLES}_stats.csv",
    alias=f"{N_SAMPLES:,} observations",
)
dataset

TabularDataset(alias='100,000 observations', path=PosixPath('data/mix_100000_stats.csv'), X_shape=(3, 100000))

In [3]:
mixtures = dataset.data
display(mixtures.head())
display(mixtures.describe().T)

,mistura1,mistura2,mistura3
0,0.268631,1.927361,2.476518
1,0.129316,0.342414,0.983813
2,0.777922,-0.048251,0.211118
3,3.408771,1.796194,0.511789
4,1.074128,-0.225014,-1.462891


,count,mean,std,min,25%,50%,75%,max
mistura1,100000.0,-0.000434,1.085589,-8.136572,-0.672978,-0.001747,0.673410,6.852251
mistura2,100000.0,0.008705,1.803989,-5.127127,-1.516551,0.011940,1.531811,5.126302
mistura3,100000.0,0.007826,2.092798,-7.698682,-1.604430,-0.000036,1.630098,9.094491


## 3. Data preparation: Centering + Whitening

Centering and whitening only need to be computed once. We reuse the resulting whitened checkpoint `Z` for both methods so they are compared on exactly the same prepared data.

In [4]:
centering = Centering()
whitening = Whitening()

preprocessing = Pipeline(steps=[centering, whitening])
Z = preprocessing.fit_transform(dataset)
Z.shape

torch.Size([3, 100000])

In [5]:
# After whitening, the covariance should be approximately the identity matrix.
torch.cov(Z)

tensor([[ 1.0000e+00, -6.0654e-07,  1.0666e-07],
        [-6.0654e-07,  1.0000e+00, -1.4659e-06],
        [ 1.0666e-07, -1.4659e-06,  1.0000e+00]])

## 4. Separate the sources

In [6]:
methods = {
    "Kurtosis": Kurtosis(),
    "Negentropy": Negentropy(),
}

separated = {}
for name, model in methods.items():
    torch.manual_seed(0)
    separated[name] = model.fit_transform(Z)

{name: Y.shape for name, Y in separated.items()}

{'Kurtosis': torch.Size([3, 100000]), 'Negentropy': torch.Size([3, 100000])}

## 5. Recovered mixing matrices

After whitening, $z = Vx_c$, where $V$ is the whitening matrix and $x_c$ contains the centered mixtures. Each method estimates an orthogonal unmixing matrix $W$, with $y = Wz$. Therefore, $x_c = V^{-1}W^Ty$, and the estimated mixing matrix is $A = V^{-1}W^T$.

In [7]:
def mixing_matrix(model):
    return torch.linalg.inv(whitening.whitening_matrix) @ model.components.T


mixing_matrices = {name: mixing_matrix(model) for name, model in methods.items()}
mixing_matrices

{'Kurtosis': tensor([[ 0.9186,  0.5582, -0.1518],
         [-0.3087,  1.7619, -0.2344],
         [-0.6283,  1.8411,  0.7715]]),
 'Negentropy': tensor([[-0.9150, -0.5656, -0.1463],
         [ 0.3216, -1.7590, -0.2382],
         [ 0.6484, -1.8366,  0.7655]])}

## 6. Separated sources

In [8]:
source_tables = {
    name: pd.DataFrame(
        Y.T.numpy(),
        columns=[f"source_{i + 1}" for i in range(Y.shape[0])],
    )
    for name, Y in separated.items()
}

for name, table in source_tables.items():
    print(name)
    display(table.head())

Kurtosis


,source_1,source_2,source_3
0,-0.305551,1.083926,0.364285
1,0.072389,0.287081,0.638868
2,0.822148,0.178998,0.505822
3,2.757880,1.429535,-0.512389
4,1.062369,-0.064550,-0.887084


Negentropy


,source_1,source_2,source_3
0,0.316831,-1.081723,0.361183
1,-0.065821,-0.288269,0.639043
2,-0.817301,-0.186223,0.511041
3,-2.749369,-1.451657,-0.495661
4,-1.068684,0.056659,-0.880010


In [9]:
# Near-zero correlations provide a simple check of decorrelation.
for name, table in source_tables.items():
    print(name)
    display(table.corr().round(4))

Kurtosis


,source_1,source_2,source_3
source_1,1.0,-0.0,0.0
source_2,-0.0,1.0,-0.0
source_3,0.0,-0.0,1.0


Negentropy


,source_1,source_2,source_3
source_1,1.0,-0.0,-0.0
source_2,-0.0,1.0,0.0
source_3,-0.0,0.0,1.0


## 7. Kurtosis vs. Negentropy

ICA does not determine the order or sign of the sources. Before comparing the methods, `align` permutes and flips the columns of one matrix to align it with the other.

In [10]:
def align(A: torch.Tensor, A_ref: torch.Tensor) -> torch.Tensor:
    """Permute and flip the columns of A to align them with A_ref."""
    A_unit = A / A.norm(dim=0, keepdim=True)
    ref_unit = A_ref / A_ref.norm(dim=0, keepdim=True)
    similarity = ref_unit.T @ A_unit
    best = similarity.abs().argmax(dim=1)
    signs = similarity.gather(1, best.unsqueeze(1)).sign().squeeze(1)
    return A[:, best] * signs


A_ref = mixing_matrices["Kurtosis"]
A_aligned = align(mixing_matrices["Negentropy"], A_ref)
max_disagreement = (A_ref - A_aligned).abs().max()
max_disagreement

tensor(0.0202)

## 8. Effect of sample size

In [11]:
def separate(model, sample_dataset):
    sample_whitening = Whitening()
    pipeline = Pipeline(steps=[Centering(), sample_whitening, model])
    Y = pipeline.fit_transform(sample_dataset)
    A = torch.linalg.inv(sample_whitening.whitening_matrix) @ model.components.T
    return Y, A


results = []
for csv_path in sorted(DATASET_DIR.glob("mix_*_stats.csv")):
    n_samples = int(csv_path.stem.split("_")[1])
    sample_dataset = TabularDataset(csv_path, alias=f"{n_samples:,} observations")

    torch.manual_seed(0)
    _, A_kurtosis = separate(Kurtosis(), sample_dataset)
    torch.manual_seed(0)
    _, A_negentropy = separate(Negentropy(), sample_dataset)

    disagreement = (A_kurtosis - align(A_negentropy, A_kurtosis)).abs().max()
    results.append({"observations": n_samples, "maximum disagreement": disagreement.item()})

pd.DataFrame(results).sort_values("observations").reset_index(drop=True)

,observations,maximum disagreement
0,100,0.151908
1,1000,0.107467
2,100000,0.020160
